# Лабораторная работа 5. Ансамбли, отбор признаков

Результат лабораторной работы − отчет. Мы предпочитаем принимать отчеты в формате ноутбуков Jupyter (ipynb-файл). Постарайтесь сделать ваш отчет интересным рассказом, последовательно отвечающим на вопросы из заданий. Помимо ответов на вопросы, в отчете так же должен быть код, однако чем меньше кода, тем лучше всем: нам − меньше проверять, вам —  проще найти ошибку или дополнить эксперимент. При проверке оценивается четкость ответов на вопросы, аккуратность отчета и кода.

### Оценивание и штрафы

* Каждая из задач имеет определенную «стоимость» (указана в скобках около задачи).
* Максимально допустимая оценка за работу — 25 балла. Из них дополнительные баллы - 12.5, обязательные - 12.5.
* Сдавать задание после указанного срока сдачи нельзя.
* «Похожие» решения считаются плагиатом и все задействованные студенты (в том числе те, у кого списали) не смогут получить за него больше 0 баллов и понижают карму.
* Если вы нашли решение какого-то из заданий в открытом источнике, необходимо прислать ссылку на этот источник (скорее всего вы будете не единственным, кто это нашел, поэтому, чтобы исключить подозрение в плагиате, необходима ссылка на источник).
* Не нужно копипастить классы между заданиями - один раз написали и потом переиспользуйте!
* Не оцениваются задания с удалёнными формулировками.
* Не оценивается лабораторная работа целиком, если она была выложена в открытый источник.

# Введение

Обратившись к [Питеру Флаху](https://www.labirint.ru/books/477617/), перечислим еще раз три ингредиента машинного обучения: задачи, модели и признаки.

В данной лабораторной работе мы возьмём предобученную нейронную сеть, входящую в набор моделей torchvision.models, и оставим от нее только "тушку": мы оставляем CNN (convolutional neural network) часть, извлекающую из изображения  признаки, и отрезаем "голову" - полносвязную сеть, которая по этим признакам классифицирует. Таким образом, мы сгенерировали для вас dataset с извлечёнными из картинок признаками. 

Как вы понимаете, нет никаких гарантий, что данный набор признаков будет полезен для нашей задачи. А значит, вы сможете:
  * применить методы отбора признаков; 
  * обучить ансамбли над деревьями решений;
  * отобрать лучшие модели.

# 0. Задача и Dataset

Данные о задаче доступны в соревновании https://www.kaggle.com/t/c5b18ffd89ec43aa95477548b54f0e7e

Метрика -- AUC ROC

In [ ]:
# SOME DATA PREPARING

# 1. Подбор гиперпараметров

Вы уже знаете, что для подбора гиперпараметров есть способ перебора по сетке. Обычно перебор некоторых значений гиперпараметров ведется по логарифмической шкале, так как это позволяет быстрее определить, какого порядка должен быть параметр, и в то же время значительно уменьшить время поиска. Последний нюанс бывает особо критичен, так как для каждого фиксированного набора гиперпараметров происходит новое обучение алгоритма и оценка качества. 

Однако такой подход к нахождению гиперпараметров является не единственно возможным. Рассмотрим более подробно, в чем может заключаться недостаток предыдущего подхода. Допустим, вам нужно подобрать 2 гиперпараметра, для каждого из которых есть сетка из 4 возможных значений. То есть всего 16 итераций обучения по сетке. Допустим также, что для оценки качества используется 5-fold CV. В итоге алгоритм будет обучен 80 раз, что уже немало. А если, например, рассмотреть случайный лес, где гиперпараметрами могут являться критерий ветвления, максимальная глубина деревьев, минимальное число объектов в листьях, максимальное число признаков, количество листьев и так далее, может получиться экспоненциально большое число обучений алгоритма, что займёт очень много времени. 

Существует несколько стратегий, позволяющих ускорить поиск гиперпараметров. Понятно, что оптимальное решение можно найти только полным перебором: все сокращённые решения гарантируют некоторое приближение к оптимуму, но не гарантируют, что найденная комбинация гиперпараметров будет самой лучшей.

Самая простая стратегия -  это **случайный** поиск по сетке. В этом случае для каждого гиперпараметра задается распределение, из которого выбираются его значения. И так как каждый раз значение каждого гиперпараметра выбирается случайно, это позволяет находить оптимум быстрее. 

Более детально о случайном поиске по сетке можно прочесть по следующим ссылкам:
 - теоретический анализ случайного поиска [Random Search for Hyper-Parameter Optimization](http://jmlr.csail.mit.edu/papers/volume13/bergstra12a/bergstra12a.pdf)
 - кратко и с юмором [Smarter Parameter Sweeps (or Why Grid Search Is Plain Stupid)](https://medium.com/rants-on-machine-learning/smarter-parameter-sweeps-or-why-grid-search-is-plain-stupid-c17d97a0e881#.pkwq17od8)
 
В sklearn случайный поиск по сетке реализован в классе [RandomizedSearchCV](http://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html#sklearn.model_selection.RandomizedSearchCV).


Более продвинутые стратегии поиска, основанные на байесовском оценивании, можно найти в специальных библиотеках - самые популярные из них [hyperopt](https://github.com/hyperopt/hyperopt) и [optuna](https://optuna.readthedocs.io/en/stable/): они реализуют похожий набор стратегий, по производительности и возможностям интеграции с другими библиотеками ML **optuna** считается предпочтительней. Кстати, обе библиотеки реализуют в том числе и случайный поиск по сетке, так что нет необходимости пользоваться реализацией *sklearn*.

**Задание 0 (3 балла). Подбор гиперпараметров.** В данной лабораторной работе вы не найдёте заданий на использование конкретной бибилотеки и конкретной стратегии. Однако в лабораторной работе вы будете обучать ансамбли моделей, которые имеют множество параметров: Random Forest, Boosting и стекинг алгоритмов (да, в стекинге можно по сетке подбирать как параметры самого стекинга, так и параметры каждого из составляющих его алгоритмов). 

Вы вольны сами решать, где и когда какую стратегию и библиотеку использовать, однако для получения 3 баллов за задание 0 вам необходимо в течение всей лабораторной работы использовать как минимум две различных стратегии любой из указанных выше библиотек (библиотеку также выбираете вы: **hyperopt** или **optuna**). Кроме того, как минимум в одном из заданий необходимо сравнение этих двух стратегий с базовым [GridSearhCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) по 
- качеству модели при найденных оптимальных параметрах;
- по скорости поиска.

Рекомендуется это проделать для последних заданий: обучение бустинга или стекинга.

Сравнение с GridSearchCV достаточно продемонстрировать только в одном из заданий, однако применение **hyperopt**, **optuna** или хотя бы **RandomizedSearchCV** из sklearn рекомендуется во всех заданиях, где нужно подбирать по сетке больше двух параметров (а вообще и для двух тоже): это сэкономит вам время.


# 2. Отбор признаков

**Задание 1 (1.5 балла). PCA.** Попробуйте снизить размерность базового набора признаков при помощи [PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html). Визуализируйте первые 2 главные компоненты, раскрасив примеры меткой класса. Подберите оптимальное число параметров по:
 - графику объясняемой дисперсии;
 - по методу Кайзера;
 - по методу сломанной трости;
 - по числу обусловленности.

Подробнее о PCA и методах оценки оптимального числа параметров см. [википедию](https://ru.wikipedia.org/wiki/%D0%9C%D0%B5%D1%82%D0%BE%D0%B4_%D0%B3%D0%BB%D0%B0%D0%B2%D0%BD%D1%8B%D1%85_%D0%BA%D0%BE%D0%BC%D0%BF%D0%BE%D0%BD%D0%B5%D0%BD%D1%82).

После выбора оптимального числа признаков обучите RandomForest на этих признаках, подберите опимальные гипарпараметры.


С одной стороны, PCA улучшает признаковое пространство тем, что приеобразует исходные признаки в максимально-некоррелированные. С другой стороны, полученные признаки неинтерпретируемы: это не даёт представление о том, какие из исходных признаков более важны. В принципе, для поставленной задачи это и не требуется, так мы и так работаем не с исходными признакми, а признакми, полученными в результате работы нейросети: эти признаки в общем случае также неинтерпретируемы. Однако в реальной жизни полезно просто ограничить размерность пространства, оставаясь во множестве исходных признаков.

Самая простая стратегия  - использовать *feature_importance_* для деревянных алгоритмов и *coeff_* для линейных.

**Задание 2 (1 балл). Univariate features importance.** Оцените важность признаков.
  * Оцените связь между целевой меткой и значением признкаков при помощи статистических тестов, воспользовавшись разделом [Univariate feature selection из sklearn](https://scikit-learn.org/stable/modules/feature_selection.html#univariate-feature-selection).
  * Отранжируйте признаки при помощи линейного классификатора с l1-регуляризацией с использованием [SelectFromModel](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectFromModel.html). Пример можно найти [здесь](https://scikit-learn.org/stable/modules/feature_selection.html#l1-based-feature-selection).
  * Отранжируйте признаки при помощи RandomForest с использованием [SelectFromModel](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectFromModel.html).

Однако что, если нам дана произвольная модель (не обязательно линейная или деревянная), и мы всё равно хотим как-то выделить подмножство важных признаков? Обычно применяется жадный алгоритм отбора признаков (можно реализовать и полный перебор, но, как вы понимаете, это будет очень затратно). Популярны две стратегии жадного отбора: снизу вверх (последовательно добавляем признаки) и сверху вних (последовательно удаляем признаки). 

На самом деле есть и другие стратегии, например, в лекциях Воронцова приводится алгоритм, который на каждом шаге может как добавлять, так и удалять признак.

В библиотеке *sklearn* жадный отбор признаков реализован в классе [SequentialFeatureSelector](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SequentialFeatureSelector.html#sklearn.feature_selection.SequentialFeatureSelector).  На каждом шаге выбор лучшего признака осуществляется на основе замера скоринговой функции на кросс-валидации.

**Задание 3 (1 балл)**. Проведите отбор признаков методом добавления, обучая те же классификаторы, что в Задании 2 (то есть какой-то линейный алгоритм и Random Forest). На отобранном множестве признаков подберите для алгоритмов наилучшие параметры. Вычислите [перестановочную оценку влияния](https://scikit-learn.org/stable/modules/generated/sklearn.inspection.permutation_importance.html#sklearn.inspection.permutation_importance) отобранных признаков.


*!дополнительно!* **Задание 4 (1.5 балла). Библиотека [shap](https://shap.readthedocs.io/en/latest/).** Обучите классификатор RandomForest на всех признаках. При помощи библиотеки shap отранжируйте признаки по степени их влияния для предсказаний модели. Совпадает ли ранжирование по shap с ранжированием на основе перестановочных оценок? Попробуйте выкинуть хвост из "невлияющих" признаков. Что произойдет с качеством? Как пересекаются множества "выкинутых" факторов и факторов, которые не были добавлены в прошлом задании?
    
    

*!дополнительно!* **Задание 5 (0.5 балла). Сравнение моделей.** Сравните модели из заданий 1-4:
- по качеству;
- модели из заданий 2-4 - по спискам признаков: что общее, чем отличаются.


# 3. Ансамбли моделей. Подбор гиперпараметров

## Бустинги

В случае бустинга композиция алгоритмов строится последовательно. Каждый следующий базовый алгоритм акцентируется на тех объектах, на которых обученная ранее композиция допускала ошибку.

Есть три наиболее популярных пакета с реализацией градиентного бустинга:  [XGBoost](https://xgboost.readthedocs.io/en/latest/), [lightGBM](https://lightgbm.readthedocs.io/en/latest/) и [CatBoost](https://catboost.ai/docs/). Есть и бругие реализации бустинга, даже в sklearn ([GradientBoostingRegressor](http://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html),  [GradientBoostingClassifier](http://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html)), однако перечисленные три реализации работают быстрее, гибче засчет большего числа параметров, в них большее внимание уделяется регуляризации и скорости.
Бустинг строит композицию из $K$ базовых алгоритмов $b_k$:

$$\hat{y}_i = \hat{y}_i^{K} = \sum_{k=1}^{K} b_k(x_i) = \hat{y}_i^{\left(K - 1\right)} + b_K(x_i), $$

минимизируя следующий функционал:

$$Obj = \sum_{i=1}^N \mathcal{L}(y_i, \hat{y}_i ) + \sum_{k=1}^{K} \Omega(b_k),$$

где
 - $N$ — размер обучающей выборки;
 - $x_i, y_i, \hat{y}_i$ — i-ый объект, правильный ответ и предсказание модели для него;
 - $\hat{y}_i^{t}$ — предсказание композиции из $t$ уже обученных базовых алгоритмов для i-го объекта;
 - $\Omega$ — регуляризатор;
 - $\mathcal{L}(y_i, \hat{y}_i)$ — функция потерь.

Функционал, оптимизируемый на $t$-ой итерации:

$$ Obj^{(t)} = \sum_{i=1}^N \mathcal{L}\left(y_i, \hat{y}_i^{(t-1)} + b_t(x_i)\right) + \Omega(b_t).$$

## 3.1 Реализация бустингов

В данной секции вам будет предложено реализовать усложненные версии бустинга над деревьями.

Для валидации ващих решений сравните вашу реализацию с одиночными деревьями и реализацией бустинга из sklearn либо из библиотек с секции 3.2 на датасете на ваш выбор.

**Задание 6 (2 балл)**. Обучение на L1 скор. Напишите реализацию градиентного бустинга, который минимизирует l1 расстояние между целевым значением и предсказанным ансамблем. Подробнее, вы можете прочитать здесь: https://explained.ai/gradient-boosting/L1-loss.html#alg:l1. Обратите внимание, что в этой реализации вам придётся "перевзвешивать" предсказания в листьях ваших деревьев. 

**Задание 7 (3 балл)**. Добавьте моменты при обучении бустинга, как описано в статье [Accelerated Gradient Boosting](https://arxiv.org/pdf/1803.02042.pdf). Сравните предложенное авторами ускорение с помощью моментами Нестерова с известными вам способамми улучшения сходимости стохастического градиентного спуска. Какие можно сделать выводы?

*дополнительно!* **Задание 8 (4 балла)**. Добавьте регуляризацию в ваше дерво. Остановитесь на регуляризации из формулы 2 статьи про [XGBoost](https://arxiv.org/pdf/1603.02754.pdf). Здесь вам придётся считать вторые производные, поэтому предлагается сначала показать, что вы считаете их правильно с помощью pytorch у простой функции. Затем уже реализуйте разбиение деревьев, которое учитывает вашу резуляризацию (части 2.1 и 2.2 статьи про xgboost).

*дополнительно!* **Задание 9 (3 балла)**. Наигравшись с бустингами для задачи бинарной классификации, давайте расширим наши бустинги для задачи многоклассовой классификации. Обратимся к оригинальной статье [Greedy Function Approximation: A Gradient Boosting Machine](https://www.cse.iitb.ac.in/~soumen/readings/papers/Friedman1999GreedyFuncApprox.pdf). Вас интересует глава **4.5**. Для проверки корректности вашей реализации проверьте алгоритм на датасете про качество вина (11 семинар)

## 3.2 Boosting libraries

В пакетах градиентного бустинга реализовано несколько различных функций потерь, что позволяет решать задачи классификации (бинарной и мультиклассовой), регрессии и ранжирования. Вот некоторые из них:

- reg:squarederror (XGBoost), RMSE (CatBoost), regression (lightGBM) — линейная регрессия;
- binary:logistic (XGBoost), LogLoss (CatBoost, только классы), CrosssEntropy (CatBoost, вероятности классов), binary (lightGBM, только классы), cross_entropy (lightGBM, вроятности классов) — логистическая регрессия;
- multi:softmax (XGBoost), MultiClass (CatBoost), multiclass (lightGBM)— softmax функция потерь для многоклассовой классификации;
- rank:pairwise (XGBoost), PairLogit, PairLogitPairwise (CatBoost), lambdarank (lightGBM) — минимизация pairwise-функции потерь для задачи ранжирования.

*!дополнительно!* **Задание 10 (1.5 балла). GBDT.**
Обучите градиентый бустинг над 100 деревьями с одинаковыми параметрами, используя любые две библиотеки из предложенных трех:  xgboost, catboost и lightGBM. Сравните получившиеся композиции, построив графики качества на обучающей и тестовой выборке в зависимости от количества деревьев. 
Подберите оптимальные гиперпараметры по сетке для обеих реализаций и сравните результаты. 

У бустинга много гиперпараметров: не обязательно перебирать все, но выберите не меньше трех, не считая количества деревьев. В качестве подсказки можете использовать ноутбук с семинара.

*!дополнительно!* **Задание 11 (0.5 балла). Сравнение ансамблей.**

Если вы сделали больше 1-го задания среди заданий 6-10, то сравните алгоритмы из этих заданий по следующим критериям:
  - время обучения в зависимости от количества деревьев,
  - качество предсказания на тестовой выборке в зависимости от количества деревьев.

### Дообучение (обучение с начальным приближением)

XGBoost, CatBoost и LightGBM позволяют задавать начальные значения отступов по всем объектам выборки, тем самым имитируя первый шаг алгоритма. По умолчанию, первый базовый алгоритм (в нашем случае - решающее дерево) строит решение на нулевых смещениях, что соответствует ситуации, когда все элементы равноправны и мы про них ничего не знаем. Как только построено первое дерево решений, второе дерево решений строится уже с учетом отступов (margin), полученных первым решающим деревом. Третье решающее дерево настраивается на отступы композиции первых двух деревьев, и т. д. XGBoost, CatBoost и LightGBM позволяют строить композицию, в которой уже первое дерево подстраивается под заранее известные отступы, которые можно получить, предварительно обучив другой алгоритм.
Для этого:

- в XGBoost у класса *DMatrix* есть методы *set_base_margin* и *get_base_margin*.
- аналогично в CatBoost у класса *Pool* есть поле *baseline* и методы *set_baseline* и *get_baseline*.
- в LightGBM у класса *Dataset* есть поле *init_score* и методы *set_init_score* и *get_init_score*, а у *lightgbm.train* есть параметр *init_model*, позволяющий продолжить обучение с уже готового экземпляра *Booster*.

Пример для XGBoost можно посмотреть в ноутбуке семинара.
[Здесь](https://catboost.ai/docs/concepts/python-usages-examples.html) можно найти пример для Catboost.
А вот пример для [LightGBM](https://github.com/Microsoft/LightGBM/blob/master/R-package/demo/boost_from_prediction.R) (правда, на языке R).


*!дополнительно!* **Задание 12 (1.5 балла). Pretraining.** Используйте лучшую композицию из задания 6 в качестве начального приближения для градиентного бустинга над деревьями (возьмите лучший бустинг из задания 7), т.е. обучайте бустинг на ошибки, которые дает композиция. Аналогичную операцию проделайте для начального приближения линейной регрессией. Дали ли подобные подходы прирост качества по сравнению с обычным градиентным бустингом и почему?

## 3.3 Стекинг
![](https://4.bp.blogspot.com/-hCxAb57kzDQ/VuMgHy3hAhI/AAAAAAAAAVk/djmL9IHv5QkLWeudjE50qDoCTbiUrTetA/s1600/Stacking.jpg)

[Stacking](https://en.wikipedia.org/wiki/Ensemble_learning#Stacking) — еще один способ объединить несколько алгоритмов в один, который часто используется как в решении реальных задач из промышленной сферы, так и в конкурсах на платформах вроде Kaggle. Подход использует понятие *базовых классификаторов*, каждый из которых независимо обучается на некотором (возможно одном и том же) множестве признаков, а также *мета-классификатора*, использующего предсказания базовых классификаторов как факторы. 


**Задание 13 (1 балл). Stacking.** Постройте стекинг ансамбль из лучших моделей [StackingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingClassifier.html#sklearn.ensemble.StackingClassifier). Подберите оптимальные гиперпараметры. Поэкспериментируйте, в частости, с гиперпараметром **passthrough**.
